**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/kalogriopoulos-2024"], check=True)
    os.chdir("/content/encore/entries/kalogriopoulos-2024")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Kalogriopoulos, Tei et al. (2024) from dandiset 001354

**Paper.** Kalogriopoulos NA, Tei R, Yan Y, Klein PM, Ravalin M, Cai B, Soltesz I, Li Y, Ting AY
(2024). Synthetic GPCRs for programmable sensing and control of cell behaviour. *Nature* 637,
230-239. [doi:10.1038/s41586-024-08282-3](https://doi.org/10.1038/s41586-024-08282-3). The claims
file (`claims/kalogriopoulos-2024.yaml`) was written from the version of record as served by
Europe PMC (PMC11666456, marked "corrected publication 2025") with its Supplementary Information.

**Dandiset.** [DANDI:001354](https://dandiarchive.org/dandiset/001354/0.250312.0036), version
0.250312.0036 (doi:10.48324/dandi.001354/0.250312.0036). The paper does not cite the dandiset or
any version of it (its Data availability says no separate datasets were generated). The archive
has one published version, 0.250312.0036, and a draft with the same 50 assets and the same size;
the published version is pinned, as the most recent published one. The dandiset holds the acute
slice recordings of Fig. 4e-j and Extended Data Fig. 12a-h: whole-cell current clamp of CA1
neurons expressing anti-mCherry PAGER-Gi, one NWB file per cell.

**What can be tested.** Every NWB file is read below. The files hold the voltage and the injected
current of every sweep, with the subject, the date and the cell, but no record of which cells
received DCZ alone and which DCZ plus mCherry, of when DCZ was applied, of each cell's mEGFP
intensity, or of the mCherry preincubation time. The slice claims compare the two conditions
around the DCZ onset, so none of them can be tested; the other 24 claims are on culture, imaging and
biochemical data that the dandiset does not hold. The notebook checks the data against the
paper's description and computes the paper's per-sweep measures and exclusions for all cells, so
that the counts the paper gives can be set against the files.

| Claim | Status |
|---|---|
| methods-slice-sample | Blocked: metadata insufficient (no condition per cell); totals shown |
| methods-mcherry-preincubation | Blocked: data missing (no preincubation times) |
| fig4i-rmp-hyperpolarization | Blocked: metadata insufficient (no condition, no DCZ onset) |
| fig4j-rheobase-increase | Blocked: metadata insufficient (no condition, no DCZ onset) |
| ed12b-input-resistance-trend | Blocked: metadata insufficient (no condition, no DCZ onset) |
| ed12cd-expression-baseline | Blocked: data missing (no mEGFP intensity) |
| ed12ef-expression-shift | Blocked: data missing (no mEGFP intensity) |
| ed12g-expression-rmp | Blocked: data missing (no mEGFP intensity) |
| 24 claims on cell culture, imaging and biochemistry (Fig. 1-3, Fig. 4b-d, k-n, Fig. 5, Extended Data Fig. 1-3, 5, 7, 13) | Blocked: data missing |

In [ ]:
import resource
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dandi.dandiapi import DandiAPIClient

import pager_slice as ps

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
cells, sweeps = ps.load_all(workers=4)
print(f"loaded {len(cells)} files, {len(sweeps)} sweeps from dandiset {ps.DANDISET_ID} "
      f"version {ps.DANDISET_VERSION} in {time.time() - started:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
client = DandiAPIClient()
meta = client.get_dandiset(ps.DANDISET_ID, ps.DANDISET_VERSION).get_raw_metadata()
summary = meta["assetsSummary"]
print("Archive description:", meta["description"].split("\n")[0])
print("Archive summary: subjects", summary.get("numberOfSubjects"), "| cells", summary.get("numberOfCells"),
      "| files", summary.get("numberOfFiles"), "| variables", summary.get("variableMeasured"))
print("Related resource:", [r.get("identifier") for r in meta.get("relatedResource", [])])
print()
cells["recorded"] = pd.to_datetime(cells.session_start)
cells["date"] = cells.recorded.dt.date.astype(str)
cells["age_days"] = (cells.recorded.dt.normalize() - pd.to_datetime(cells.date_of_birth)).dt.days
cells["duration_min"] = cells.n_sweeps * 5 / 60
by_subject = cells.groupby("subject").agg(cells=("cell_id", "size"), dates=("date", lambda d: ", ".join(sorted(set(d)))),
                                          sex=("sex", lambda s: ",".join(sorted(set(s)))),
                                          date_of_birth=("date_of_birth", "first"),
                                          age_days=("age_days", lambda a: f"{a.min()}-{a.max()}"),
                                          sweeps_min=("n_sweeps", "min"), sweeps_max=("n_sweeps", "max"))
print("Cells per subject:")
print(by_subject.to_string())
print(f"\nTotal: {cells.subject.nunique()} subjects, {len(cells)} cells, {cells.slice_id.nunique()} slice ids, "
      f"{cells.cell_id.nunique()} cell ids; slice id equal to cell id in {int((cells.slice_id == cells.cell_id).sum())} files")
print("Recording length per cell (min): ", cells.duration_min.describe().round(1).to_dict())
print("Targeted layer:", cells.targeted_layer.value_counts().to_dict())

The archive's summary (8 subjects, 50 cells) agrees with the files: 8 mice, all female (as the
Methods state), 50 cells, each in its own file and with its own slice id. The paper counts 15
cells from 7 mice with DCZ and 26 cells from 6 mice with DCZ plus mCherry: 41 cells, and 13 mice
if no mouse gave cells to both groups. The files hold 9 cells more, and 8 mice can hold the
paper's 7 and 6 only if at least 5 mice gave cells to both groups. The Methods allow that
("Recordings in DCZ alone were avoided after >1 DCZ + mCherry experiment per day"), so mice
were shared across conditions. Ages at recording are compared with the Methods (injected "when 4-8 weeks old", recorded "3-6
weeks following initial injection") in the next cell, under two readings: whole weeks (49 to 98
days) and up to the end of the eighth and sixth weeks (28 to 62 days at injection, 21 to 48 days
later, so 49 to 110 days).

In [ ]:
cells["in_49_98"] = cells.age_days.between(28 + 21, 56 + 42)
cells["in_49_110"] = cells.age_days.between(28 + 21, 62 + 48)
print("Age at recording per subject (days), and whether it lies in 49-98 days (whole weeks) or 49-110 days "
      "(to the end of the eighth and sixth weeks):")
print(cells.groupby("subject").agg(age_min=("age_days", "min"), age_max=("age_days", "max"),
                                   in_49_98=("in_49_98", "all"), in_49_110=("in_49_110", "all")).to_string())
print()
print("Sampling rate (Hz):", sorted(cells.rate_hz.unique()), "| samples per sweep:", sorted(cells.n_samples.unique()),
      "| sweep length (s):", sorted((cells.n_samples / cells.rate_hz).unique()))
steps = sweeps.groupby("cell_id").t_start_s.apply(lambda t: sorted(set(np.round(np.diff(t.values), 3))))
print("Interval between sweep starts (s), over all cells:", sorted({x for s in steps for x in s}))
print("Units stored: channel 0", sorted(cells.ch0_unit.unique()), "| channel 1", sorted(cells.ch1_unit.unique()),
      "| stimulus", sorted(cells.stim_unit.unique()))
print("Series descriptions:", sorted(cells.stim_description.unique()))
print("Stored stimulus levels (pA), first sweep:", sorted({tuple(x) for x in cells.stim_levels_pa}))
print("Fraction of equal neighbouring samples in channel 0, first sweep (pairs 2k,2k+1 against 2k+1,2k+2): "
      f"median {cells.pairs_identical.median():.3f} against {cells.pairs_identical_shift.median():.3f}")
print("Residual SD when one half of the samples of channel 0, first sweep, is fitted on the 8 samples of the other half "
      "on each side (ADC counts):")
print("  odd from even:", cells.interp_resid_odd.describe().round(3).to_dict())
print("  even from odd:", cells.interp_resid_even.describe().round(3).to_dict())
print("Files whose smaller residual is below 0.5 counts:",
      int((cells[["interp_resid_odd", "interp_resid_even"]].min(axis=1) < 0.5).sum()))
print("Same test, smaller residual, on the even samples of the same sweeps upsampled back to 20 kHz and rounded:")
for name in ["linear", "cubic", "fft"]:
    print(f"  {name}:", cells[f"upsampled_resid_{name}"].describe().round(3).to_dict())

The files are sampled at 20 kHz, where the Methods state digitization at 10 kHz. Neighbouring
samples are equal as often within the pairs (2k, 2k+1) as across them, so the 20 kHz samples are
not repeated 10 kHz samples. That test alone cannot exclude interpolation from 10 kHz, so each
half of the samples was also fitted, by least squares, on the 16 nearest samples of the other
half. On the even samples of the same sweeps upsampled back to 20 kHz by linear, cubic or
band-limited (FFT) interpolation and rounded, the smaller residual is near the rounding error
of the integer samples (0.30 to 0.57 counts, printed above); in the stored data both
residuals lie between 0.87 and 1.58 counts in every file and are nearly equal. The test uses the
first sweep of each file and these three interpolators; an upsampling it does not model, such as
one that adds noise, is not excluded. The result is consistent with acquisition at 20 kHz, and
the measures below use the stored rate. Each file is one continuous run of 5-s sweeps with no gap
between sweeps (every interval between sweep starts is 5 s), so the time of a sweep within the
recording is its index times 5 s. The stimulus series holds only four levels (-150, -100, 0 and
+500 pA): it gives the ramp as a step to +500 pA. The eight mice were 53 to 107 days old at
recording by their dates of birth. Two, PK-109 and PK-110, at 101 and 107 days, are older than
whole weeks would allow (98 days) but within the 110 days the Methods allow if "4-8 weeks old" and
"3-6 weeks" run to the end of the eighth and sixth weeks, so the ages are consistent with the
Methods.


In [ ]:
v = sweeps
print("Channel 0 (membrane potential, mV), per-sweep RMP over all sweeps:", v.rmp_mv.describe().round(1).to_dict())
print("Channel 1 (stored in volts), mean over the step, read as pA:", v.i_step_pa.describe().round(2).to_dict())
print("Channel 1 before any injection, read as pA:", v.i_baseline_pa.describe().round(2).to_dict())
print("Channel 1 between the step and 1.0 s, read as pA:", v.i_gap_pa.describe().round(2).to_dict())
print("Channel 1 from 1.01 to 1.09 s, before the ramp, read as pA:", v.i_pre_ramp_pa.describe().round(2).to_dict())
v = v.assign(ramp_end=v.ramp_end_s.round(2))
ramps = v.groupby("ramp_end").agg(cells=("cell_id", "nunique"), sweeps=("sweep", "size"),
                                  slope_pa_s=("ramp_slope_pa_s", "median"), slope_min=("ramp_slope_pa_s", "min"),
                                  slope_max=("ramp_slope_pa_s", "max"), start_pa=("ramp_start_pa", "median"),
                                  peak_pa=("ramp_peak_pa", "median"))
print("Delivered ramp in channel 1 by end time (s): fitted slope (pA/s), fitted start at 1.1 s and peak (pA):")
print(ramps.round(1).to_string())
per_cell = v.groupby("cell_id").agg(n=("ramp_end", "nunique"), end=("ramp_end", "first"), date=("cell_id", "first"))
print("Cells whose ramp differs between sweeps:", int((per_cell.n > 1).sum()))
short = per_cell[per_cell.end < 3.0]
print("Cells with a ramp ending before 3.1 s:", len(short), short.index.tolist())
print("Ramp current at the first action potential (A1), channel 1 against the fitted ramp of the sweep, pA:",
      (v.rheo_A1_pa - v.rheo_fit_A1_pa).describe().round(2).to_dict())
nominal = v.ramp_end > 3.0
print("Same against the nominal ramp formula (-150 to +500 pA over 1.1 to 3.1 s), cells with a ramp ending at 3.1 s, pA:",
      (v.rheo_A1_pa - v.rheo_formula_A1_pa)[nominal].describe().round(2).to_dict())
early = v[v.t_ap_A1 < 1.2]
print("Sweeps whose first action potential (A1) comes in the first 100 ms of the ramp:", len(early),
      "in cells", sorted(early.cell_id.unique()), "; rheobase there (pA):", early.rheo_A1_pa.round(1).tolist())
print("Stored stimulus at the first action potential (A1), pA:", sorted(v.rheo_stored_A1_pa.dropna().unique()))
print()
print("Clock check: step onset in channel 1 (s):", sorted(v.i_step_onset_s.round(5).unique()))
print("Voltage change across the step onset (2-5 ms after minus 3 ms before), mV:", v.dv_step_onset_mv.describe().round(2).to_dict())
print("Same measure 2 to 8 ms before the onset (no step), mV:", v.dv_before_onset_mv.describe().round(2).to_dict())
print("Sweeps whose voltage falls by less than 1 mV at the step onset:", int((v.dv_step_onset_mv > -1).sum()), "of", len(v))
first = v[v.sweep == 0].set_index("cell_id")
print("\nRMP of the first sweep (Methods: holding current set so each cell began at -65 mV), mV:",
      first.rmp_mv.describe().round(1).to_dict())
print("Cells whose first-sweep RMP is within 3 mV of -65 mV:", int(first.rmp_mv.between(-68, -62).sum()), "of", len(first))
base = v.groupby("cell_id").i_baseline_pa.agg(["min", "max", "std"])
print("\nChannel 1 before any injection, per cell: within-cell range (max - min, pA)",
      (base["max"] - base["min"]).describe().round(3).to_dict(), "| within-cell SD (pA)", base["std"].describe().round(3).to_dict())

Channel 1 is stored as a membrane potential series in volts, but it reads -99.8 pA during the
-100 pA step, -149.9 pA from 1.0 to 1.1 s, and then a ramp: it is the injected current, in
amperes. The current is held at -150 pA for 100 ms before the ramp, which the Methods do not
mention. In 40 cells the ramp runs from -150 pA at 1.1 s to +500 pA at 3.1 s, as the Methods
state (-150 to +500 pA across 2 s). In the 10 cells listed above (eight of the nine recorded on
2024-08-07, one on 2024-07-24 and one on 2024-07-26) it reaches +500 pA at 2.7 s or 2.5 s, a
steeper ramp over 1.6 s or 1.4 s, the same in every sweep of the cell. The stored stimulus shows
none of this: it is the same step to +500 pA in every file. Rheobase is therefore read from
channel 1, where it agrees with the fitted ramp of each sweep and, in the 40 cells with the
nominal ramp, with the Methods' ramp formula, both printed above. A cell's rheobase depends on
the ramp's slope, so the ten cells with a steeper ramp are not comparable with the others on
rheobase without correction; in a within-cell change the slope is constant. Five sweeps fire in the
first 100 ms of the ramp, at -147.5 to -112.3 pA (printed above); the rule takes their first
action potential as it comes. A test of rheobase would have to decide whether these sweeps count,
as they would pull a cell's rheobase down.

Before any injection channel 1 reads 0.47 to 0.56 pA, so either no holding current was applied or
the channel does not include it; the Methods say holding current was adjusted so that each cell
began at -65 mV, and 49 of 50 cells have a first-sweep RMP within 3 mV of -65 mV. Within a cell
the channel 1 baseline varies by at most a few hundredths of a pA (printed above), so if channel
1 includes the holding current, the holding current is near zero and constant; if it does not,
the files hold no record of it. Whether the holding current was constant, which the criterion of
fig4j-rheobase-increase asks the author to show, therefore cannot be shown from the files. Channel 1 steps
down at the same sample (0.50005 s) in every sweep, and the voltage falls across that moment
(median -7.1 mV; 1 of 12858 sweeps by less than 1 mV) and not in the 6 ms before it (median 0.0
mV), so voltage and current share the sweep's clock. The sweep clock cannot be related to a drug
application, as no application time is recorded (next cell).

In [ ]:
d = sweeps.rheo_A2_pa - sweeps.rheo_A1_pa
both = sweeps.t_ap_A1.notna() & sweeps.t_ap_A2.notna()
for label, sel in [("all sweeps", sweeps.rmp_mv.notna()), ("sweeps with RMP at or below -50 mV", sweeps.rmp_mv <= ps.RMP_MAX_MV)]:
    a1, a2 = sweeps.t_ap_A1.notna() & sel, sweeps.t_ap_A2.notna() & sel
    print(f"{label} ({int(sel.sum())}): first action potential under A1 {int(a1.sum())}, under A2 {int(a2.sum())}, "
          f"A2 only {int((a2 & ~a1).sum())}, A1 only {int((a1 & ~a2).sum())}")
ok = both & (sweeps.rmp_mv <= ps.RMP_MAX_MV)
print("Rheobase A2 - A1 per sweep where both detect, pA; all sweeps:", d[both].describe().round(2).to_dict())
print("  sweeps with RMP at or below -50 mV:", d[ok].describe().round(2).to_dict())
print("Sweeps with RMP at or below -50 mV where A2 comes more than 10 pA before A1:", int((d[ok] < -10).sum()),
      "| more than 10 pA after A1:", int((d[ok] > 10).sum()), "| within 1 pA:", int((d[ok].abs() <= 1).sum()), "of", int(ok.sum()))
for rule in ps.AP_RULES:
    t = sweeps[f"t_ap_{rule}"]
    early = (t < ps.RAMP[0] + 0.1) & (sweeps.rmp_mv <= ps.RMP_MAX_MV)
    print(f"Sweeps with RMP at or below -50 mV whose first {rule} event falls within 100 ms of the ramp start "
          f"(1.1 to 1.2 s): {int(early.sum())} in {sweeps.cell_id[early].nunique()} cells")
off = sweeps[ok & (d < -10)]
print("Cells holding the sweeps where A2 precedes A1 by more than 10 pA:", off.cell_id.value_counts().to_dict())
print("Mean rheobase per cell over these sweeps, A2 minus A1 (pA):",
      sweeps[ok].groupby("cell_id").apply(lambda g: (g.rheo_A2_pa - g.rheo_A1_pa).mean()).describe().round(2).to_dict())
a2_only = sweeps.t_ap_A2.notna() & sweeps.t_ap_A1.isna() & (sweeps.rmp_mv <= ps.RMP_MAX_MV)
print("Peak voltage in the 3 ms after the first event (mV): A1, sweeps with RMP at or below -50 mV:",
      sweeps.peak_A1_mv[sweeps.rmp_mv <= ps.RMP_MAX_MV].describe().round(1).to_dict())
print("  A2, sweeps where A2 comes more than 10 pA before A1:", off.peak_A2_mv.describe().round(1).to_dict())
print("  A2, sweeps A2 alone detects:", sweeps.peak_A2_mv[a2_only].describe().round(1).to_dict(),
      "| their cells:", sweeps.cell_id[a2_only].value_counts().head(8).to_dict())
early_a2 = (sweeps.t_ap_A2 < ps.RAMP[0] + 0.1) & (sweeps.rmp_mv <= ps.RMP_MAX_MV)
print("  A2, sweeps whose first A2 event falls in the first 100 ms of the ramp:", sweeps.peak_A2_mv[early_a2].round(1).tolist())


A2 (dV/dt above 20 V/s) and A1 (upward crossing of 0 mV) are compared above, sweep by sweep.
Among the 11839 sweeps with RMP at or below -50 mV, the two rules give a rheobase within 1 pA of
each other in 10335 of the 10675 sweeps where both detect an event, and A2 detects an event in
456 sweeps where A1 detects none. Where A2 comes more than 10 pA earlier (335 sweeps, 325 of them
in cells 2024_08_07_0022 and 2024_08_07_0012), its first event peaks at a median of -15.2 mV, and
the events A2 alone detects peak between -19.7 and -0.1 mV: these are fast spikes that do not
reach 0 mV, which A1 misses, and they are concentrated in a few cells (printed above). In the
first 100 ms of the ramp, where the cell is held near -150 pA, A2 finds 7 first events; 5 peak
above 0 mV, like the 5 of A1, and 2 peak at -49.3 and -55.8 mV, which are not action potentials:
there A2 picks up a fast deflection. The mean rheobase per cell differs between the rules by a
median of 0.14 pA, and by up to 336 pA in one cell. No claim is tested, so this decides no
outcome; if the conditions were known, rheobase under A2 and A1 would differ in these few cells.


In [ ]:
keywords = ["dcz", "mcherry", "deschloroclozapine", "drug", "pharmac", "condition", "treat",
            "bath", "wash", "egfp", "gfp", "intensity", "preincub", "incubat", "onset", "epoch", "note",
            "interval", "trial"]
hits = {}
for _, c in cells.iterrows():
    for line in c.text + c.paths:
        low = line.lower()
        # the NeuroConv source path names the converter's own files, not the experiment
        if "source_script@file_name" in low:
            continue
        for k in keywords:
            if k in low:
                hits.setdefault(k, set()).add(line[:120])
print("Every group and dataset in the first file (specifications excepted; sweep numbers replaced by NN):")
for path in cells.paths.iloc[0]:
    print("  /" + path)
print("Files with the same set of paths as the first:", int(sum(p == cells.paths.iloc[0] for p in cells.paths)), "of", len(cells))
print("Top-level groups in any file:", sorted({p.split("/")[0] for ps_ in cells.paths for p in ps_ if p}))
print("\nText values per file (attributes of every object, every sweep's series included, and every string "
      "dataset of any length), distinct after replacing sweep numbers:", int(np.median([len(t) for t in cells.text])))
print("Text values or paths that name a drug, condition, timing, interval or expression level:", hits if hits else "none")
print("Distinct values, over all sweeps of all files, of the fields that could hold one:")
for field in ["session_description", "general/experimenter", "general/lab", "general/institution",
              "general/intracellular_ephys/electrode-0/description", "general/intracellular_ephys/electrode-1/description",
              "sequential_recordings/stimulus_type", "@description", "@stimulus_description", "@comments"]:
    if field.startswith("@"):   # an attribute of the sweep series
        keep = lambda path: path.startswith(("acquisition", "stimulus")) and path.endswith(field)
    else:                       # a dataset's value
        keep = lambda path: "@" not in path and path.endswith(field)
    vals = sorted({line.split(": ", 1)[1] for t in cells.text for line in t if keep(line.split(": ", 1)[0])})
    print(f"  {field}: {vals}")


No file names a drug, a condition, a time of application, an expression level or a
preincubation. The scan reads every attribute of every object in every file, all sweeps
included, and every string dataset of any length, including the text columns of the
intracellular recording tables. No file has an /intervals group (no epochs, trials or time
intervals), and /analysis and /processing are empty (path list above). Every sweep's series
carries the same description ("Response to: ramp" or "Stim type: ramp"), the stimulus
description "N/A" and the comment "no comments"; the session description is the same generic
sentence in every file, and the experimenter, lab and institution are "_". The dandiset metadata
names the two conditions only in its description. Without a condition per cell and a DCZ onset
per recording, the comparisons of Fig. 4i,j and Extended Data Fig. 12b cannot be made.


In [ ]:
dup = cells.groupby("hash_first_sweep").cell_id.apply(list)
print("Files sharing an identical first sweep:", [d for d in dup if len(d) > 1] or "none")
per_cell = sweeps.groupby("cell_id").agg(rmp_mean=("rmp_mv", "mean"), rin_mean=("rin_last100_mohm", "mean"),
                                         n=("sweep", "size")).round(3)
same = per_cell[per_cell.duplicated(keep=False)]
print("Cells with identical sweep count, mean RMP and mean Rin:", same.index.tolist() or "none")
starts = cells.session_start.value_counts()
print("Session start times used by more than one file:", starts[starts > 1].to_dict() or "none")

No recording is deposited twice: no two files share a first sweep, a session start, or the same
sweep count with the same mean RMP and Rin.

## Measures and exclusions for the blocked slice claims

The claims are blocked, so this section tests none of them. It computes the per-sweep measures
and the exclusions of the Methods (as `pager_slice` documents, with the readings the criterion of
fig4i-rmp-hyperpolarization names) for all 50 cells, so that the number of cells the paper's rules
keep can be set against the paper's 41 (reading (b) of methods-slice-sample), and shows the time
course of each cell. Which cells belong to which condition, and where DCZ was applied, the files
do not say.

> methods-slice-sample: The slice data comprise 15 cells from 7 mice recorded with DCZ alone and
> 26 cells from 6 mice recorded with DCZ plus mCherry.

In [ ]:
rows = []
for rin_window in ps.RIN_WINDOWS:
    for rule in ps.AP_RULES:
        c, s = ps.apply_exclusions(cells, sweeps, rin_window, rule)
        kept = c[c.cell_kept]
        sk = s[s.cell_id.isin(kept.cell_id)]
        with_sweeps = sk[sk.sweep_kept].cell_id.nunique()
        rows.append({"rin_window": rin_window, "ap_rule": rule,
                     "cells_kept": len(kept), "mice_kept": kept.subject.nunique(),
                     "cells_excluded_rin": int((~c.cell_kept).sum()),
                     "kept_cells_with_included_sweeps": with_sweeps,
                     "sweeps_total": len(s), "sweeps_of_kept_cells": len(sk),
                     "dropped_rmp_above_-50": int((sk.rmp_mv > ps.RMP_MAX_MV).sum()),
                     "then_dropped_no_ap": int(((sk.rmp_mv <= ps.RMP_MAX_MV) & sk[f"t_ap_{rule}"].isna()).sum()),
                     "sweeps_included": int(sk.sweep_kept.sum())})
counts = pd.DataFrame(rows)
print("Reading (a), all cells in the dataset:", len(cells), "cells from", cells.subject.nunique(), "mice")
print("Reading (b), cells passing the initial Rin exclusion (first sweep, Rin <= 500 megaohms):")
print(counts.to_string(index=False))
c, _ = ps.apply_exclusions(cells, sweeps, "last100", "A1")
print("\nFirst-sweep Rin (last 100 ms of the step), megaohms:", c.rin_first_mohm.describe().round(1).to_dict())
print("Cells per subject, all and kept (last 100 ms window):")
print(c.groupby("subject").agg(cells=("cell_id", "size"), kept=("cell_kept", "sum")).T.to_string())
print("\nPaper: 15 + 26 = 41 cells; 7 and 6 mice. Per-condition counts cannot be formed: no condition is recorded.")
totals = sorted({len(cells)} | set(counts.cells_kept))
print("Totals under the readings:", totals, "| paper's total: 41 | excess:", [t - 41 for t in totals],
      "| a split of any of these totals into two groups matching 15 and 26:", any(t == 41 for t in totals))

**Result: blocked.** Under reading (a) the dataset has 50 cells from 8 mice; under reading (b),
with either Rin window and either detection rule, one cell (from PK-113, first-sweep Rin 525
megaohms) is excluded and 49 cells from 8 mice remain, against the paper's 41 cells in total from
7 and 6 mice. Neither reading can be split into the two conditions, because no file records which
condition a cell was in, so the four counts the claim states cannot be formed. The totals alone
already rule out "reproduced" under both readings: no split of 50 or 49 cells gives 15 and 26.
This assumes that every deposited cell was recorded in one of the two conditions, as the
dandiset's description names only these two; if some cells were in neither, 41 of them could
still split as 15 and 26.
Had the conditions been recorded, the outcome could only have been partially or not reproduced,
depending on the mouse counts. The 8 or 9 extra cells are not explained by any exclusion the
paper states.


In [ ]:
c, s = ps.apply_exclusions(cells, sweeps, "last100", "A1")
s = s.merge(cells[["cell_id", "date", "session_start"]], on="cell_id")
s["t_min"] = s.t_start_s / 60
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
for cid, g in s.groupby("cell_id"):
    g = g[g.sweep_kept]
    axes[0].plot(g.t_min, g.rmp_mv, lw=0.6)
    axes[1].plot(g.t_min, g.rin_last100_mohm, lw=0.6)
    axes[2].plot(g.t_min, g.rheo_A1_pa, lw=0.6)
axes[0].set_ylabel("RMP (mV)"); axes[1].set_ylabel("Rin, last 100 ms (megaohms)"); axes[2].set_ylabel("rheobase A1 (pA)")
axes[2].set_xlabel("time from the start of the recording (min)")
axes[1].set_ylim(0, 500)
fig.suptitle("Included sweeps of the kept cells (no DCZ onset is recorded)")
plt.tight_layout(); plt.show()
inc = s[s.sweep_kept]
for col, ax in zip(["rmp_mv", "rin_last100_mohm", "rheo_A1_pa"], axes):
    lo, hi = ax.get_ylim()
    print(f"{col}: range {inc[col].min():.1f} to {inc[col].max():.1f}; points outside the axes: "
          f"{int(((inc[col] < lo) | (inc[col] > hi)).sum())}")
print("Recording length of kept cells (min):", (c[c.cell_kept].n_sweeps * 5 / 60).describe().round(1).to_dict())
order = cells.sort_values("session_start").groupby("date").cell_id.apply(list)
print("\nCells in recording order per day (the Methods record DCZ-alone cells only before a second "
      "DCZ plus mCherry cell on a day):")
for day, ids in order.items():
    print(" ", day, ids)

The time courses show each cell over a recording of 15.5 to 26 minutes. Without the
application time, a change "upon 2-min DCZ addition" cannot be measured, and without the
condition the change cannot be compared between groups. The recording order per day is printed
because the Methods tie the DCZ-alone recordings to the start of a day; that rule does not say
which cells were DCZ alone, so it is not used to assign conditions.

## Summary

| Claim | Outcome |
|---|---|
| methods-slice-sample | Blocked: metadata insufficient. 50 cells from 8 mice; no condition per cell |
| methods-mcherry-preincubation | Blocked: data missing. No preincubation times in the files |
| fig4i-rmp-hyperpolarization | Blocked: metadata insufficient. No condition per cell, no DCZ onset |
| fig4j-rheobase-increase | Blocked: metadata insufficient. No condition per cell, no DCZ onset |
| ed12b-input-resistance-trend | Blocked: metadata insufficient. No condition per cell, no DCZ onset |
| ed12cd-expression-baseline | Blocked: data missing. No mEGFP intensity or image per cell |
| ed12ef-expression-shift | Blocked: data missing. No mEGFP intensity or image per cell |
| ed12g-expression-rmp | Blocked: data missing. No mEGFP intensity or image per cell |
| 24 claims on culture, imaging and biochemical data | Blocked: data missing. Not in the dandiset, and the paper deposited no other data |

How the analysis differs from the paper's: no claim was tested. The per-sweep measures follow the
Methods with the readings the claims file names; rheobase is read from channel 1, the current
monitor, because the stored stimulus gives the ramp as a step. The two detection rules give a
rheobase within 1 pA in 10335 of 10675 sweeps; they differ in a few cells whose spikes do not
reach 0 mV, and A2 takes 2 small deflections early in the ramp for action potentials (data
check).

What was found about the dataset:

- No file records the condition of a cell (DCZ alone or DCZ plus mCherry), the time DCZ was
  applied, the mEGFP intensity of the cell, or the mCherry preincubation time. No file has an
  intervals group or notes or pharmacology fields, and the session description is generic.
- The files hold 50 cells from 8 mice; the paper reports 41 cells (15 and 26) from 7 and 6
  mice. The extra cells are not marked, and the cells the paper's Rin rule keeps are counted
  above. Neither total can give the paper's 15 and 26.
- Channel 1 of every sweep is stored as a CurrentClampSeries in volts but holds the injected
  current in amperes.
- The stimulus series gives the ramp (-150 to +500 pA over 2 s) as a step to +500 pA; only
  channel 1 shows the ramp. In 10 of 50 cells the delivered ramp reaches +500 pA after 1.6 s or
  1.4 s, where the Methods state 2 s, and the stimulus series does not show it. Every sweep holds
  -150 pA for 100 ms before the ramp, which the Methods do not describe.
- The paper's initial Rin exclusion keeps 49 of the 50 cells, not the 41 the paper analysed; the
  files do not mark which cells the paper left out, or why.
- The files are sampled at 20 kHz; the Methods state 10 kHz. The samples are not repeated, and
  they do not behave like 10 kHz samples upsampled by linear, cubic or band-limited
  interpolation, which is consistent with acquisition at 20 kHz.
- Whether the holding current was constant cannot be shown: channel 1 reads about 0.5 pA before
  any injection, so it either does not include the holding current or none was applied.
- Experimenter, lab and institution are "_", and the electrodes and device have no description.